In [1]:
import sys
import subprocess
import os

# --- 1. 尝试导入 (解决名字混淆问题) ---
print("🚀 正在尝试加载 Pinocchio 库...")

try:
    # 核心修正：虽然包名叫 'pin'，但导入时必须用 'pinocchio'
    import pinocchio as pin
    print(f"🎉 成功导入 'pinocchio' (别名 pin) ! 版本: {pin.__version__}")
    
except ImportError:
    print("⚠️ 'import pinocchio' 失败，尝试检查安装路径...")
    
    # 如果导入失败，可能是路径没加进去，手动加一下
    # 根据你的日志，库在 /usr/local/lib/python3.10/dist-packages
    global_path = "/usr/local/lib/python3.10/dist-packages"
    if global_path not in sys.path:
        sys.path.append(global_path)
        print(f"已手动添加搜索路径: {global_path}")
    
    try:
        import pinocchio as pin
        print(f"🎉 重试后成功导入! 版本: {pin.__version__}")
    except ImportError as e:
        print(f"❌ 依然失败. 错误信息: {e}")
        print("请尝试在下方的终端(Terminal)里运行: sudo chmod -R 755 /usr/local/lib/python3.10/dist-packages")

# --- 2. 补全其他画图库 ---
def install(package):
    try:
        __import__(package)
    except ImportError:
        print(f"📦 正在补充安装: {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, 
                               "-i", "https://pypi.tuna.tsinghua.edu.cn/simple", "--user"])

install("plotly")
install("scipy")
install("pandas")
install("matplotlib")

# --- 3. 最终环境确认 ---
import numpy as np
import plotly.graph_objects as go

print("-" * 30)
if 'pin' in locals():
    print("✅ 环境完美！后续代码请直接使用 'pin' 来调用函数。")
    print(f"例如: model = pin.buildModelFromUrdf(...)")
else:
    print("❌ 环境依然有问题，请联系指导老师或检查 Docker 权限。")

🚀 正在尝试加载 Pinocchio 库...
🎉 成功导入 'pinocchio' (别名 pin) ! 版本: 3.8.0
------------------------------
✅ 环境完美！后续代码请直接使用 'pin' 来调用函数。
例如: model = pin.buildModelFromUrdf(...)


In [2]:
# --- 配置区域 ---

# 1. 你的 URDF 文件路径
urdf_path = "/anyverse/sub_modules/isaacros/src/wheeled_humanoid/robot_model/urdf/wheel_robot.urdf"

# 2. 是否强制使用虚拟模型 (调试用)
USE_DUMMY_ROBOT = False 

# ----------------

print(f"🔍 正在寻找模型文件: {urdf_path}")

if USE_DUMMY_ROBOT or not os.path.exists(urdf_path):
    if not USE_DUMMY_ROBOT:
        print(f"⚠️ 警告: 找不到路径 {urdf_path}")
    print("🛠️ 将构建一个【虚拟 7轴机械臂】用于演示流程...")
    
    # 使用 pin.buildSampleModelManipulator()
    model = pin.buildSampleModelManipulator() 
    
    # 这里的 model.nq 是自由度，如果生成的不是7轴，我们可以不管，先跑通流程
else:
    print(f"✅ 找到文件，正在加载...")
    # 关键修改：使用 pin.buildModelFromUrdf (不再是 pinocchio.)
    model = pin.buildModelFromUrdf(urdf_path)

# 创建数据对象 (用于存储计算过程中的速度、加速度、位置结果)
data = model.createData()

print("-" * 30)
print(f"✅ 模型加载成功！")
print(f"📊 关节数量 (njoints): {model.njoints} (包含基座)")
print(f"🔧 自由度 (nq): {model.nq}")

# 如果自由度 > 7 (比如双臂)，我们后续代码需要适配
if model.nq > 7:
    print("💡 提示：检测到这是一个多自由度机器人(如双臂)，后续采样代码可能需要稍作调整。")

# --- 调试代码：查找真实的 Frame 名字 ---
print(f"当前模型总共有 {model.nframes} 个 Frame。")

# 打印所有名字包含 'left' 或 'gripper' 的 Frame，帮你缩小范围
print("\n=== 可能是左手的 Frame ===")
for frame in model.frames:
    if "left" in frame.name or "L_" in frame.name: # 根据命名习惯过滤
        print(f"ID: {model.getFrameId(frame.name)} | Name: {frame.name}")

print("\n=== 可能是末端(gripper)的 Frame ===")
for frame in model.frames:
    if "gripper" in frame.name or "hand" in frame.name or "tip" in frame.name:
        print(f"ID: {model.getFrameId(frame.name)} | Name: {frame.name}")

🔍 正在寻找模型文件: /anyverse/sub_modules/isaacros/src/wheeled_humanoid/robot_model/urdf/wheel_robot.urdf
✅ 找到文件，正在加载...
------------------------------
✅ 模型加载成功！
📊 关节数量 (njoints): 33 (包含基座)
🔧 自由度 (nq): 32
💡 提示：检测到这是一个多自由度机器人(如双臂)，后续采样代码可能需要稍作调整。
当前模型总共有 108 个 Frame。

=== 可能是左手的 Frame ===
ID: 4 | Name: chassis_left_Joint
ID: 5 | Name: chassis_left_Link
ID: 32 | Name: left_fixed
ID: 33 | Name: AR5_5_07L_base
ID: 34 | Name: AR5_5_07L_joint_1
ID: 35 | Name: AR5_5_07L_link1
ID: 36 | Name: AR5_5_07L_joint_2
ID: 37 | Name: AR5_5_07L_link2
ID: 38 | Name: AR5_5_07L_joint_3
ID: 39 | Name: AR5_5_07L_link3
ID: 40 | Name: AR5_5_07L_joint_4
ID: 41 | Name: AR5_5_07L_link4
ID: 42 | Name: AR5_5_07L_joint_5
ID: 43 | Name: AR5_5_07L_link5
ID: 44 | Name: AR5_5_07L_joint_6
ID: 45 | Name: AR5_5_07L_link6
ID: 46 | Name: AR5_5_07L_joint_7
ID: 47 | Name: AR5_5_07L_link7
ID: 48 | Name: AR5_5_07L_tcp_joint
ID: 49 | Name: AR5_5_07L_tcp
ID: 50 | Name: left_flange_base_joint
ID: 51 | Name: left_flange_link
ID: 52 | Name: l

In [ ]:
import numpy as np
import pinocchio as pin
from tqdm import tqdm
import plotly.graph_objects as go

# --- 1. 手动指定左右手的名字 (请根据你的 URDF 修改!) ---
# 根据你之前的日志，你的末端名字里应该包含 gripper
# 请务必核对你的模型，这里假设是 left_gripper_base_joint 和 right_gripper_base_joint
name_left = "left_gripper_base_joint"   # <--- 请核对
name_right = "right_gripper_base_joint" # <--- 请核对

print(f"🎯 双臂分析模式:")
print(f"   左手: {name_left}")
print(f"   右手: {name_right}")

try:
    id_left = model.getFrameId(name_left)
    id_right = model.getFrameId(name_right)
except:
    print("❌ 错误：找不到指定的 Frame 名字，请检查 print(model.frames) 的输出")
    raise ValueError("Frame name not found")


def get_chain_q_indices(model, frame_name, stop_at_root=True):
    """
    自动回溯运动链，获取从 Root 到 Frame 路径上所有关节的 q 索引。
    """
    if not model.existFrame(frame_name):
        raise ValueError(f"Frame '{frame_name}' 不存在！")
        
    frame_id = model.getFrameId(frame_name)
    joint_id = model.frames[frame_id].parent
    
    chain_joint_ids = []
    
    while joint_id > 0: # 0 是 Universe
        if stop_at_root and model.joints[joint_id].shortname() == "JointModelFreeFlyer":
            break
        chain_joint_ids.append(joint_id)
        joint_id = model.parents[joint_id]
        
    q_indices = []
    for jid in chain_joint_ids:
        idx_q = model.joints[jid].idx_q
        nq = model.joints[jid].nq
        if idx_q >= 0 and nq > 0:
            q_indices.extend(list(range(idx_q, idx_q + nq)))
            
    return set(q_indices)

def get_pure_arm_indices(model, target_arm_name, other_arm_name):
    """
    通过集合运算，自动提取【纯手臂】关节索引。
    逻辑：(目标手全路径) - (另一只手全路径) = 纯目标手臂
    这样会自动剔除 腰、胸、头、基座 等公共父节点。
    """
    idx_target = get_chain_q_indices(model, target_arm_name)
    idx_other = get_chain_q_indices(model, other_arm_name)
    
    # 差集运算：属于 Target 但不属于 other 的关节
    pure_indices = list(idx_target - idx_other)
    pure_indices.sort()
    
    return pure_indices




# --- 2. 采样设置 ---
# %%


# ==========================================
# 1. 定义 IK 求解器 
# ==========================================
def solve_ik_topology_locked(model, data, frame_id, target_pos, target_rot, 
                             q_neutral, active_indices, 
                             q_init=None, max_iter=50, eps=1e-3):
    """
    拓扑锁定 IK 求解器。
    
    :param q_neutral: 标准站立姿态（用于重置被锁定的关节）
    :param active_indices: 允许动的关节索引列表 (由拓扑分析得出)
    """
    if q_init is None:
        q = q_neutral.copy()
    else:
        q = q_init.copy()
        
    dt = 1e-1
    damp = 1e-6
    success = False

    # 将 active_indices 转为 numpy array 方便索引
    active_mask = np.array(active_indices)

    for i in range(max_iter):
        pin.forwardKinematics(model, data, q)
        pin.updateFramePlacements(model, data)

        oMf = data.oMf[frame_id]
        err_pos = oMf.translation - target_pos
        err_rot = pin.log3(oMf.rotation.T @ target_rot) 
        err = np.concatenate([err_pos, err_rot])

        if np.linalg.norm(err) < eps:
            success = True
            break

        J = pin.computeFrameJacobian(model, data, q, frame_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        
        # 求解速度 v
        v = - J.T @ np.linalg.solve(J @ J.T + damp * np.eye(6), err)
        
        # 更新关节 q
        q_next = pin.integrate(model, q, v * dt)
        
        # ---【核心逻辑：拓扑锁定】---
        # 1. 只有 active_indices 里的值允许更新
        # 2. 其他所有值强制重置为 q_neutral (保持不动)
        # 这种写法比 q[0:7]=0 更通用，它能锁死腰部、头、甚至非当前手臂
        
        # 先创建一个全量副本
        q_new = q_neutral.copy()
        # 只把允许动的部分，从计算结果里抄过来
        # 注意：这里假设 active_indices 对应的是 q 的直接索引
        for idx in active_indices:
            q_new[idx] = q_next[idx]
            
        q = q_new
        
        # 简单的限位处理 (仅处理允许动的关节)
        q = np.clip(q, model.lowerPositionLimit, model.upperPositionLimit)

    return success, q

# ==========================================
# 2. 准备采样
# ==========================================

# --- A. 准备工作 ---
# 确保 model, data, name_left, name_right 已经定义好了
# name_left = "left_gripper_base_joint" (你的真实名字)
# name_right = "right_gripper_base_joint"

print("🧠 1. 正在进行拓扑分析...")
# 自动计算纯左臂关节
active_indices_L = get_pure_arm_indices(model, name_left, name_right)
print(f"   左臂独享关节索引 (q_idx): {active_indices_L}")
print(f"   (已自动锁死: 基座、腰部、头、右手)")

# 准备标准站立姿态 (所有非动关节将保持这个姿态)
q_stand = pin.neutral(model)

# --- B. 生成体素网格 ---
# 既然是纯左臂，我们把搜索范围集中在左侧 (Y > 0)
x_range = np.linspace(0.1, 0.9, 15)   # 前后
y_range = np.linspace(0.0, 0.8, 15)   # 向左
z_range = np.linspace(0.4, 1.3, 15)   # 上下

X, Y, Z = np.meshgrid(x_range, y_range, z_range)
voxel_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

# --- C. 设定目标姿态 ---
# 假设保持与站立时相同的手掌姿态
pin.forwardKinematics(model, data, q_stand)
pin.updateFramePlacements(model, data)
target_rot_L = data.oMf[id_left].rotation.copy()

print(f"\n🚀 2. 开始 IK 遍历 (搜索点数: {len(voxel_points)})")

valid_points = []
valid_mani = []

# 热启动猜测值
q_guess = q_stand.copy()

for target_pos in tqdm(voxel_points, desc="Topology IK"):
    
    is_ok, q_sol = solve_ik_topology_locked(
        model, data, id_left, 
        target_pos, target_rot_L, 
        q_neutral=q_stand,      # 锁定参考姿态
        active_indices=active_indices_L, # 只允许左臂动！
        q_init=q_guess
    )
    
    if is_ok:
        # 再次确认：位置误差真的足够小 (防止数值假阳性)
        pin.forwardKinematics(model, data, q_sol)
        pin.updateFramePlacements(model, data)
        real_pos = data.oMf[id_left].translation
        if np.linalg.norm(real_pos - target_pos) < 0.02: # 2cm 容差
            valid_points.append(target_pos)
            
            # 计算灵巧度 (只考虑手臂关节的雅可比)
            # 这是一个高级技巧：如果只算 active joints 的灵巧度会更准
            # 但简单起见，算整体也可，因为其他部分不动
            pin.computeJointJacobians(model, data, q_sol)
            J = pin.getFrameJacobian(model, data, id_left, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)[:3, :]
            w = np.sqrt(np.linalg.det(J @ J.T))
            valid_mani.append(w)
            
            q_guess = q_sol # 更新热启动

valid_points = np.array(valid_points)


# --- D. 可视化 ---
print(f"\n✅ 分析完成！有效点数: {len(valid_points)}")

if len(valid_points) > 0:
    fig = go.Figure(data=[
        go.Scatter3d(
            x=valid_points[:, 0],
            y=valid_points[:, 1],
            z=valid_points[:, 2],
            mode='markers',
            marker=dict(size=4, color=valid_mani, colorscale='Viridis', opacity=0.6,
                       colorbar=dict(title="Manipulability")),
            name="纯手臂工作空间"
        ),
        # 画个基座原点参考
        go.Scatter3d(x=[0], y=[0], z=[0], mode='markers', marker=dict(color='black', size=5), name="Base")
    ])
    
    fig.update_layout(
        title="基于拓扑锁定的定向可达性 (只动左臂，锁死腰部/基座)",
        scene=dict(aspectmode='data')
    )
    fig.write_html("topology_ik_left.html")
    fig.show()
else:
    print("❌ 未找到可达点。请检查搜索范围或目标姿态。")

/tmp/ipykernel_4181108/2274877938.py:32: UserWarning: Deprecated member. Use Frame.parentJoint instead.
  joint_id = model.frames[frame_id].parent


🎯 双臂分析模式:
   左手: left_gripper_base_joint
   右手: right_gripper_base_joint
🧠 1. 正在进行拓扑分析...
   左臂独享关节索引 (q_idx): [6, 7, 8, 9, 10, 11, 12]
   (已自动锁死: 基座、腰部、头、右手)

🚀 2. 开始 IK 遍历 (搜索点数: 3375)


Topology IK: 100%|██████████| 3375/3375 [00:03<00:00, 917.84it/s]


✅ 分析完成！有效点数: 0
❌ 未找到可达点。请检查搜索范围或目标姿态。


In [4]:
# %%
# --- 5. 可视化定向工作空间 ---
import plotly.graph_objects as go

if len(valid_points) == 0:
    print("❌ 没有找到任何可达点！可能原因：")
    print("1. R_target 设置得不合理（比如让手反关节扭曲）。")
    print("2. 搜索范围 (x_range, y_range) 设到了机器人摸不到的地方。")
    print("3. IK 迭代次数太少。")
else:
    print(f"🎨 正在绘制定向可达性图谱...")
    
    # 散点图
    scatter = go.Scatter3d(
        x=valid_points[:, 0],
        y=valid_points[:, 1],
        z=valid_points[:, 2],
        mode='markers',
        marker=dict(
            size=5,             # 体素点可以画大一点
            color=valid_mani,   # 颜色代表灵巧度
            colorscale='Viridis', # 换个颜色区分之前的图
            opacity=0.6,
            colorbar=dict(title="灵巧度 (IK解)")
        ),
        name=f'定向可达区'
    )

    # 绘制未达到的网格点（可选，用淡灰色显示，方便看这一块是不是真的摸不到）
    # 如果点太多可能会卡，这里取一部分示例
    # unreached_mask = ... (略，为了性能暂不画)

    # 布局
    fig = go.Figure(data=[scatter])
    
    # 画一个箭头表示当前的“目标姿态”是什么方向
    # 比如画一个从原点出发的小坐标系
    
    fig.update_layout(
        title=f"定向可达性分析 (Oriented Reachability)<br><sub>约束: 固定末端姿态 | Arm: Left</sub>",
        scene=dict(
            xaxis_title='X (前向)',
            yaxis_title='Y (横向)',
            zaxis_title='Z (高度)',
            aspectmode='data'
        ),
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()
    fig.write_html("oriented_reachability_left.html")

❌ 没有找到任何可达点！可能原因：
1. R_target 设置得不合理（比如让手反关节扭曲）。
2. 搜索范围 (x_range, y_range) 设到了机器人摸不到的地方。
3. IK 迭代次数太少。


In [ ]:
import sys
import subprocess
import os
import numpy as np
import pinocchio as pin # 确保 pin 已导入

# --- 1. 自动安装 meshcat ---
try:
    import meshcat
    import meshcat.geometry as g
except ImportError:
    print("🔧 安装 meshcat...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "meshcat", 
                           "-i", "https://pypi.tuna.tsinghua.edu.cn/simple"])
    import meshcat
    import meshcat.geometry as g

from pinocchio.visualize import MeshcatVisualizer

# --- 2. 准备几何模型 (修复路径问题) ---
# 这里的逻辑是：URDF 里的 package://robot_model/ 对应的是硬盘上的 .../wheeled_humanoid/robot_model/
# 所以我们需要把 .../wheeled_humanoid/ 这个路径加到搜索列表里

dir_urdf = os.path.dirname(urdf_path) # .../urdf
dir_pkg = os.path.dirname(dir_urdf)   # .../robot_model
dir_root = os.path.dirname(dir_pkg)   # .../wheeled_humanoid <--- 关键！

# 构建搜索路径列表
mesh_dirs = [dir_root, dir_pkg, dir_urdf, "/anyverse"]

print("🎨 正在加载机器人 3D 模型...")

try:
    # 重新构建模型，这次带上几何搜索路径
    model, collision_model, visual_model = pin.buildModelsFromUrdf(
        urdf_path, 
        mesh_dirs, 
        pin.JointModelFreeFlyer() if model.nq > 7 else None
    )
    
    # 初始化 Visualizer
    viz = MeshcatVisualizer(model, collision_model, visual_model)
    
    # 强制开启 MeshCat 服务
    # url_type="tcp" 确保绑定到本地端口
    viz.initViewer(open=False) 
    viz.loadViewerModel()
    
    print("✅ 机器人模型加载成功")

except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    print("⚠️ 依然使用简易模式...")
    viz = None

# --- 3. 渲染点云 ---
if viz is not None:
    viewer = viz.viewer
    
    # 颜色映射
    if len(manipulability) > 0:
        colors = np.zeros((3, len(points)))
        max_m = np.max(manipulability)
        colors[0, :] = manipulability / max_m # R
        colors[2, :] = 1.0 - (colors[0, :])   # B
    else:
        colors = np.array([1, 0, 0])

    # 添加点云
    viewer["reachability_cloud"].set_object(
        g.PointCloud(position=points.T, color=colors, size=0.01)
    )
    
    # 显示零位姿态
    viz.display(pin.neutral(model))

    print("-" * 30)
    print("🌐 可视化服务已启动，请进行端口转发操作！")

🎨 正在加载机器人 3D 模型...
You can open the visualizer by visiting the following URL:
http://127.0.0.1:7006/static/
✅ 机器人模型加载成功！(不再是圆柱体了)


NameError: name 'manipulability' is not defined